# Visualizing Volatility Prediction

Author: Pete King

We visualize the distribution of mean absolute error (MAE) for each ETF, comparing mean and median MAE for our LSTM model using "limited" features against the predictions of our baseline model (VIX).  We use units native to the VIX (annualized volatility) rather than realized volatility, since most financial professionals think in terms of annualized volatility.

In [1]:
#123456789012345678901234567890123456789012345678901234567890123456789012345678
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import altair as alt

In [2]:
with open('volatility_results.json') as f:
    results = json.load(f)

In [3]:
trials = list(results.keys())
num_trials = len(trials)
num_trials

30

In [4]:
etfs = list(results[trials[0]].keys())
etfs

['BIL',
 'BND',
 'GLD',
 'HYG',
 'IEF',
 'IWM',
 'LQD',
 'QQQ',
 'SPY',
 'TIP',
 'TLT',
 'XLB',
 'XLE',
 'XLF',
 'XLI',
 'XLK',
 'XLP',
 'XLRE',
 'XLU',
 'XLV',
 'XLY']

In [5]:
models = list(results[trials[0]][etfs[0]])
models

['Ridge', 'GBR', 'LSTM']

In [6]:
feature_sets = list(results[trials[0]][etfs[0]][models[0]])
feature_sets

['limited', 'full', 'PCA']

In [7]:
splits = list(results[trials[0]][etfs[0]][models[0]][feature_sets[0]])
splits

['train', 'val', 'test']

In [8]:
# For three LSTM models ("limited," "full," and "PCA"), collect model MAE of 
# future volatility predictions on the testing set for each ETF as a DataFrame
rows = []
for etf in etfs:
    for feature_set in feature_sets:
        for trial in trials:
            row = [etf, 'LSTM', feature_set, 'test', trial]
            # Add each trial as a row vector
            row.append(
                results[trial][etf]['LSTM'][feature_set]['test']['MAE']
                * np.sqrt(252) * 100
            )
            rows.append(row)
        MAE = pd.DataFrame(
            rows,
            columns=[
                'ETF', 'Model Type', 'Feature Set', 'Split', 'Trial', 'MAE (ann vol)'
            ]
        )
MAE

,ETF,Model Type,Feature Set,Split,Trial,MAE (ann vol)
0,BIL,LSTM,limited,test,1,0.526203
1,BIL,LSTM,limited,test,2,0.019822
2,BIL,LSTM,limited,test,3,0.449551
3,BIL,LSTM,limited,test,4,0.885897
4,BIL,LSTM,limited,test,5,0.267994
...,...,...,...,...,...,...
1885,XLY,LSTM,PCA,test,26,2.625596
1886,XLY,LSTM,PCA,test,27,5.365024
1887,XLY,LSTM,PCA,test,28,4.352419
1888,XLY,LSTM,PCA,test,29,3.050710


In [9]:
# Get baseline (VIX) prediction error (MAE) for each ETF
with open('volatility_baseline.json') as f:
    baseline = json.load(f)

In [10]:
baseline_MAE = {}
for etf in baseline.keys():
    baseline_MAE[etf] = baseline[etf]['test'] * np.sqrt(252) * 100
baseline_MAE

{'BIL': np.float64(0.17563962908988645),
 'BND': np.float64(4.23339412564217),
 'GLD': np.float64(18.704471736609623),
 'HYG': np.float64(4.204085096625274),
 'IEF': np.float64(5.319727631465403),
 'IWM': np.float64(20.68229242397633),
 'LQD': np.float64(6.21677410396111),
 'QQQ': np.float64(19.190237420942157),
 'SPY': np.float64(14.85420080710729),
 'TIP': np.float64(3.919100207761235),
 'TLT': np.float64(11.83044444820734),
 'XLB': np.float64(17.03389560991618),
 'XLE': np.float64(21.369555569522642),
 'XLF': np.float64(16.816825951252344),
 'XLI': np.float64(15.945859428436565),
 'XLK': np.float64(22.778110498439645),
 'XLP': np.float64(12.429302807655278),
 'XLRE': np.float64(15.691955360984558),
 'XLU': np.float64(15.472949365282668),
 'XLV': np.float64(15.359399196625342),
 'XLY': np.float64(21.126291101113875)}

In [11]:
# Prepare to display 21 ETFs in a 3x7 grid
etf_rows = []
for i in range(7):
    etf_row = []
    for j in range(3):
        etf_row.append(etfs[i * 3 + j])
    etf_rows.append(etf_row)
etf_rows

[['BIL', 'BND', 'GLD'],
 ['HYG', 'IEF', 'IWM'],
 ['LQD', 'QQQ', 'SPY'],
 ['TIP', 'TLT', 'XLB'],
 ['XLE', 'XLF', 'XLI'],
 ['XLK', 'XLP', 'XLRE'],
 ['XLU', 'XLV', 'XLY']]

In [16]:
# Base chart for each ETF
base = alt.Chart(MAE).mark_boxplot().encode(
    alt.X('MAE (ann vol):Q'),
    alt.Y('Feature Set:N'),
    alt.Color('Feature Set:N')
).properties(
    width=200,
    height=100,
)
# Ruler mark for baseline MAE

# Make 7x3 grid to compare model MAE with different feature sets vs. baseline
chart = alt.vconcat()
for i in range(7):
    chart_row = alt.hconcat()
    for etf in etf_rows[i]:
        baseline = alt.Chart().mark_rule(strokeDash=[2, 2]).encode(
            x=alt.datum(baseline_MAE[etf]),
            color=alt.value('red')
        )
        chart_row |= base.transform_filter(
            alt.datum.ETF == etf
        ).properties(title=etf) + baseline
    chart &= chart_row
chart

alt.VConcatChart(...)

The small multiples chart allows us to see that our models consistently achieve a smaller MAE than the baseline model (VIX) on the test set.  We can also see that a LSTM model using the 'limited' or 'PCA' feature sets achieve a smaller median MAE for the majority of ETFs compared to that achieved by a LSTM model using the 'full' feature set.
 - For 11/21 ETFs, an LSTM model using the 'PCA' feature set achieved a lower median MAE than a LSTM model using the 'limited' feature set
 - For 10/21 ETFs, an LSTM model using the 'limited' feature set achieved a lower median MAE than a LSTM model using the 'PCA' feature set